In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

In [2]:
# Define a sparse autoencoder
class SparseAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.encoder = nn.Linear(input_dim, hidden_dim)
        self.decoder = nn.Linear(hidden_dim, input_dim)

    def forward(self, x):
        z = F.relu(self.encoder(x))
        x_hat = self.decoder(z)
        return x_hat, z

def train_sae(X_np, hidden_dim=1024, l1_lambda=1e-3, num_epochs=20, batch_size=64, lr=1e-3):
    # Convert input to torch tensor
    X = torch.tensor(X_np, dtype=torch.float32)
    dataset = TensorDataset(X)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # Init model
    model = SparseAutoencoder(input_dim=X.shape[1], hidden_dim=hidden_dim)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    # Training loop
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for (batch,) in dataloader:
            optimizer.zero_grad()
            recon, hidden = model(batch)
            recon_loss = criterion(recon, batch)
            sparsity_loss = l1_lambda * hidden.abs().mean()
            loss = recon_loss + sparsity_loss
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(batch)
        print(f"Epoch {epoch+1}: Loss = {total_loss / len(dataset):.4f}")

    return model

In [3]:
clevr_embeddings = torch.load('Embeddings/CLEVR/CLIP_patch_embeddings_percentthrumodel_100.pt')
X_np = clevr_embeddings.numpy()

AttributeError: 'dict' object has no attribute 'numpy'

In [ ]:
print(X_np.shape)